# Paper figures --- editable source

Every figure in `paper/ska_drift/main.tex`, with the **actual source** of its
plotting function inlined so colours, labels and layout can be tuned here.

**How this works.** Each cell below holds one `fig_*` function copied verbatim
from `ska_hi_plots.py` or `ska_hi_design.py`. Running a cell redefines that
function *in the notebook*, shadowing the module version, and the call at the
bottom of the cell writes the PNG to `figures/`. The paper picks it up with no
copy step --- `\graphicspath` searches `../../figures/` first.

**Two sources of truth.** That is the cost of this notebook. Once a cell is
edited it diverges from the `.py`, and whichever ran last owns the PNG. When a
tuning is settled, paste it back into the script; until then, treat the
notebook as the live version and re-run it after any script change.

**VSCode.** Close this notebook before anything edits it on disk, or VSCode
will write its in-memory copy back over the change.

**Venv.** Run under `/home/geoff/gibbs_venv_312`, not `limTOD/.venv` ---
`ska_hi_plots` builds an HI cube and needs `pyccl`.


## Setup

In [ ]:
import os, sys
sys.path.insert(0, "/home/geoff/limTOD/examples/SKA")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

import ska_hi_plots as P
import ska_hi_design as D
# brings in the palette, NMODES, RESDIR, collect, load_bands, load_realisations,
# freq_mode_density, BANDS, SEED_RUNS ... i.e. everything the pasted bodies use.
from ska_hi_design import *
# private helpers, which `import *` will not bring across
from ska_hi_plots import _tidy, _save, _title, _figsize, _load

%matplotlib inline
# Inline DISPLAY size only. Saved resolution is set by _save(), which uses
# STYLE["dpi"] (default 200) -- change that, not this, to alter the PNGs:
#     P.STYLE["dpi"] = 300
plt.rcParams["figure.dpi"] = 110
print("figures ->", P.FIGDIR, "| save dpi =", P.STYLE.get("dpi", 200))

## Palette

The one place to change colours. `set_palette()` pushes the values into this
notebook **and** into both modules, so helpers like `_tidy` (grid colour) and
`_title` follow.

In [ ]:
PALETTE = dict(
    DRIFT    = "#2a78d6",   # strategy 1
    RASTER   = "#eb6834",   # strategy 2
    THIRD    = "#1baf7a",   # third arm (drift8 / drift12 / 1-f)
    SURFACE  = "#fcfcfb",   # marker fill / figure background
    INK      = "#0b0b0b",   # primary text
    INK2     = "#52514e",   # panel titles
    MUTED    = "#898781",   # annotations
    GRID     = "#e1e0d9",
    BASELINE = "#c3c2b7",
)

def set_palette(**kw):
    """Update the palette everywhere: notebook globals and both modules."""
    PALETTE.update(kw)
    for name, value in PALETTE.items():
        globals()[name] = value
        for mod in (P, D):
            if hasattr(mod, name):
                setattr(mod, name, value)
    return PALETTE

set_palette()

## Data

Read once from `results/`. Nothing here recomputes --- these are all cached
`.npz` files, so the whole notebook runs in seconds.

In [ ]:
exp, abl, rk = _load()          # experiment 006, its ablation, its rank study

lad      = np.load(os.path.join(RESDIR, "hi_ladder_f350_ns64.npz"))
lad_long = np.load(os.path.join(RESDIR, "hi_ladder_long_ns64.npz"))
pv       = np.load(os.path.join(RESDIR, "hi_priorvar_tk_f350_400_nc32_ns64.npz"))
matched  = np.load(os.path.join(RESDIR, "hi_ablation_matched_f350_400_nc32_ns64.npz"))
mech     = np.load(os.path.join(RESDIR, "hi_ablation_mech_f350_400_nc32_ns64.npz"))
modes    = np.load(os.path.join(RESDIR, "hi_modes_f350_400_nc32_ns64.npz"))

d = collect(matched, lad, modes, freq_mode_density(), extra_abl=mech)

# the published drift is 3 strips, drift12 is 12 -- the only two geometries
# with a measured HI floor, so the only overlay points fig_ladder can take
floors = {n: d[k]["floor"] for n, k in ((3, "drift"), (12, "drift12")) if k in d}
bands  = load_bands()
vals   = load_realisations()

for s_ in ("drift", "drift8", "drift12", "raster"):
    if s_ in d:
        print(f"{s_:8s} {d[s_]['dens']:6.2f} modes/100deg^2   floor {d[s_]['floor']:7.1f}x")

## `hi_ladder_geometry`

*Sec. 2.3 -- Scan strategies.* The ladder as sky geometry. Below one beam the strips merge and concentrate measurement; at one beam and beyond they separate and buy area instead.

Source: `ska_hi_design.fig_ladder_geometry`. Every `fig_*` takes `text=` to override the title/subtitle without touching the body, e.g.
`fig_ladder_geometry(lad, text=('My title', 'My subtitle'))`.

In [ ]:
def fig_ladder_geometry(ld=None, text=None):
    """The ladder as geometry, for explaining the mode-count result to others.

    Parked due north, a dish at elevation ``el`` points at declination
    ``dec = 90 + phi - el`` (phi = -30.713 at the Karoo), so elevation steps are
    declination steps one-for-one, and each strip is a band of constant Dec
    swept through 15 deg of RA by one hour of Earth rotation.

    Top row: the footprints. Bands are drawn one FWHM thick and semi-transparent,
    so overlap darkens. Bottom row: the same thing as a cross-scan cut -- the
    beam response against Dec, individual strips light, their sum bold. The sum
    is the honest picture of what the ladder does: tight spacing gives a narrow,
    smoothly covered band; wide spacing gives separated ribbons with troughs
    between them, which is the regime where the strips have become independent
    surveys and the ladder has stopped buying information.

    The bands show FWHM only. The map-maker keeps every pixel above 5 per cent
    of beam peak, which reaches considerably further, so the areas quoted from
    ``ska_hi_ladder.py`` are larger than these rectangles -- they are annotated
    from the measurement, never derived from the drawing.
    """
    from matplotlib.patches import Rectangle

    lat, fwhm = -30.713, 3.983
    ra_span = 15.0                       # one hour of drift
    sigma = fwhm / 2.3548

    # (n_strips, spacing_beams) -- the spacing arm, plus one point of the
    # count arm so both levers appear in the same picture.
    panels = [(3, 0.25), (3, 0.5), (3, 1.0), (3, 1.5), (8, 0.5)]
    meas = {}
    if ld is not None:
        for n, sp, d, a in zip(ld["n_strips"], ld["spacing_beams"],
                               ld["dens"], ld["area"]):
            meas[(int(n), round(float(sp), 2))] = (float(d), float(a))

    fig, axes = plt.subplots(2, len(panels), figsize=P._figsize(
        "hi_ladder_geometry", (15.0, 6.4)),
        gridspec_kw=dict(height_ratios=[1.25, 1.0], hspace=0.38, wspace=0.22))

    decs_all = []
    for n, sp in panels:
        els = EL_CENTRE_ = 50.0 + (np.arange(n) - (n - 1) / 2.0) * sp * fwhm
        decs_all.append(90.0 + lat - els)
    dlo = min(d.min() for d in decs_all) - fwhm
    dhi = max(d.max() for d in decs_all) + fwhm

    grid = np.linspace(dlo, dhi, 900)
    ymax = max(np.sum([np.exp(-0.5 * ((grid - d) / sigma) ** 2) for d in decs],
                      axis=0).max() for decs in decs_all)

    for col, ((n, sp), decs) in enumerate(zip(panels, decs_all)):
        # --- top: footprints on the sky ---------------------------------
        ax = axes[0, col]; P._tidy(ax)
        for d in decs:
            ax.add_patch(Rectangle((0, d - fwhm / 2), ra_span, fwhm,
                                   facecolor=DRIFT, edgecolor="none",
                                   alpha=0.42, zorder=3))
            ax.plot([0, ra_span], [d, d], color=SURFACE, lw=0.8, zorder=4)
        ax.set_xlim(-1.0, ra_span + 1.0); ax.set_ylim(dlo, dhi)
        ax.set_xlabel("RA drift  [deg]", fontsize=8.6)
        if col == 0:
            ax.set_ylabel("declination  [deg]")
        else:
            ax.set_yticklabels([])
        lab = f"{n} strips, {sp:g} beam" + ("" if sp == 1 else "s")
        if (n, sp) == (3, 0.5):
            lab += "\n(published 52/50/48)"
        elif (n, sp) == (8, 0.5):
            lab += "\n(8 dishes)"
        ax.set_title(lab, color=INK, fontsize=9.4, fontweight="semibold",
                     loc="left", pad=6, linespacing=1.35)

        # --- bottom: the cross-scan cut ---------------------------------
        ax = axes[1, col]; P._tidy(ax)
        tot = np.zeros_like(grid)
        for d in decs:
            g = np.exp(-0.5 * ((grid - d) / sigma) ** 2)
            tot += g
            ax.plot(grid, g, color=DRIFT, lw=1.0, alpha=0.40, zorder=3)
        ax.plot(grid, tot, color=DRIFT, lw=2.4, zorder=5)
        ax.set_xlim(dlo, dhi); ax.set_ylim(0, ymax * 1.10)
        ax.set_xlabel("declination  [deg]", fontsize=8.6)
        if col == 0:
            ax.set_ylabel("summed beam response")
        else:
            ax.set_yticklabels([])
        # Only report a trough where one exists. Below ~0.5 beams the sum is
        # a single peak and "trough depth" measured between the outer strip
        # centres is meaningless -- it reads as if tighter packing were worse.
        lo = np.r_[False, (tot[1:-1] < tot[:-2]) & (tot[1:-1] < tot[2:]), False]
        inner = lo & (grid > decs.min()) & (grid < decs.max())
        if n > 1 and inner.any():
            note = f"dips to {tot[inner].min() / tot.max():.2f} of peak"
        elif n > 1:
            note = "no dip — strips fully merged"
        else:
            note = "single strip"
        ax.annotate(note, (0.5, 0.94), xycoords="axes fraction", ha="center",
                    color=MUTED, fontsize=8.0)
        if (n, round(sp, 2)) in meas:
            d_, a_ = meas[(n, round(sp, 2))]
            ax.annotate(f"{d_:.1f} modes/100 deg$^2$\n{a_:.0f} deg$^2$ measured",
                        (0.5, 0.74), xycoords="axes fraction", ha="center",
                        color=INK, fontsize=8.6, fontweight="semibold",
                        linespacing=1.4)

    P._title(fig, "The declination ladder: overlapping strips concentrate "
             "information, separated strips just add sky",
             "Parked due north, $\\delta = 90\\degree + \\phi - \\mathrm{el}$, so "
             "elevation steps are declination steps one-for-one, and one hour of "
             "Earth rotation sweeps each strip through 15$\\degree$ of RA. Bands "
             "are one FWHM thick; the map-maker keeps everything above 5% of beam "
             "peak, so measured areas (annotated) exceed the rectangles.",
             x=0.012, y_title=1.035, y_sub=0.985, override=text)
    fig.tight_layout(rect=(0, 0, 1, 0.90))
    return P._save(fig, "hi_ladder_geometry")


fig_ladder_geometry(lad)

## `hi_priorvar`

*Sec. 3 -- Map-making, `Prior choice'.* Loosening the prior raises N_eff and worsens the residual at the same time.

Source: `ska_hi_design.fig_priorvar`. Every `fig_*` takes `text=` to override the title/subtitle without touching the body, e.g.
`fig_priorvar(pv, text=('My title', 'My subtitle'))`.

In [ ]:
def fig_priorvar(pv, d=None, nmodes=NMODES, text=None):
    """Does the trade space survive the prior, or is it a picture of the prior?

    Left: the residual against the prior variance, floor and total, both
    strategies. Right: the same sweep drawn as a *track* across the trade
    space, against the two-point power law the design figure draws through the
    published anchors. The question the right panel asks is the only one that
    matters for the design figure:

      * a track that runs ALONG the power law means the curve is a statement
        about the prior. Loosening the prior would then buy you residual for
        free, and the chart is recommending a prior, not a survey.
      * a track that runs ACROSS it means the mode count and the residual are
        separately real, and the curve is a statement about the survey.

    ``d`` is the design-figure dict from ``collect``; passing it draws the
    published anchors and the power law for reference.
    """
    scales = np.asarray(pv["scales"], float)
    fig, (axA, axB) = plt.subplots(
        1, 2, figsize=P._figsize("hi_priorvar", (13.4, 5.6)),
        gridspec_kw=dict(width_ratios=[1.0, 1.0], wspace=0.26))

    # --- left: residual against the prior scale ----------------------------
    P._tidy(axA)
    for s, style in STRAT.items():
        axA.plot(scales, pv[f"{s}_total_{nmodes}"], color=style["color"],
                 marker="o", ms=7, mfc=SURFACE, mew=2.0, zorder=4)
        axA.plot(scales, pv[f"{s}_floor_{nmodes}"], color=style["color"],
                 ls=(0, (4, 2)), lw=1.6, alpha=0.75, zorder=3)
        axA.annotate(s, (scales[-1], pv[f"{s}_total_{nmodes}"][-1]),
                     textcoords="offset points", xytext=(9, 0), va="center",
                     color=style["color"], fontsize=9.5, fontweight="semibold")
    axA.axhline(1.0, color=INK, lw=1.4, zorder=2)
    axA.annotate("HI level", (scales[0], 1.0), textcoords="offset points",
                 xytext=(2, 6), color=INK, fontsize=8.8)
    axA.axvline(1.0, color=MUTED, lw=1.0, ls=(0, (1, 2)), zorder=1)
    axA.annotate("as published", (1.0, axA.get_ylim()[1]),
                 textcoords="offset points", xytext=(4, -10), color=MUTED,
                 fontsize=8.4, rotation=90, va="top")
    axA.set_xscale("log"); axA.set_yscale("log")
    axA.set_xlim(scales.min() * 0.5, scales.max() * 3.0)
    axA.set_xlabel("prior variance / truth patch variance   "
                   "(right = looser prior)")
    axA.set_ylabel(f"post-clean residual / HI   ({nmodes} modes removed)")
    axA.legend(handles=[
        Line2D([], [], color=MUTED, marker="o", ms=7, mfc=SURFACE, mew=2.0,
               label="total"),
        Line2D([], [], color=MUTED, ls=(0, (4, 2)), lw=1.6,
               label="beam + prior floor alone")],
        loc="lower right", bbox_to_anchor=(1.0, 0.07), fontsize=8.8)
    axA.set_title("What the prior's amplitude is worth", color=INK2,
                  fontsize=9.6, loc="left", pad=6)

    # --- right: the same sweep as a track across the trade space -----------
    P._tidy(axB)
    axB.set_xscale("log"); axB.set_yscale("log")
    bx0, bx1, by0, by1 = 7.0, 95.0, 3.0, 6e3
    axB.set_xlim(bx0, bx1); axB.set_ylim(by0, by1)
    if d is not None:
        # Clipped to the panel: unclipped it spans five decades and flattens
        # everything else. Both published anchors lie ON it by construction --
        # it is the line through them -- so what the eye is being asked to
        # judge is the ANGLE the prior tracks cross it at.
        xs = np.geomspace(bx0, bx1, 400)
        ys = d["raster"]["ratio"] * (xs / d["raster"]["dens"]) ** d["slope"]
        keep = (ys > by0) & (ys < by1)
        axB.plot(xs[keep], ys[keep], color=MUTED, lw=1.6, ls=(0, (5, 3)),
                 zorder=2)
        axB.annotate("two-point power law from the\ndesign figure "
                     f"(slope ${d['slope']:.1f}$)",
                     (bx0 * 1.05, 2600), color=MUTED, fontsize=8.4, ha="left",
                     va="center", linespacing=1.45)
    lab_off = {"drift": (0, -30, "center", "top"),
               "raster": (12, -12, "left", "top")}
    for s_, style in STRAT.items():
        x = np.asarray(pv[f"{s_}_dens"], float)
        y = np.asarray(pv[f"{s_}_total_{nmodes}"], float)
        axB.plot(x, y, color=style["color"], lw=2.2, zorder=4)
        axB.plot(x, y, color=style["color"], ls="none", marker="o", ms=6,
                 mfc=SURFACE, mew=1.8, zorder=5)
        j = int(np.argmin(np.abs(scales - 1.0)))
        axB.plot([x[j]], [y[j]], color=style["color"], marker="o", ms=13,
                 mec=SURFACE, mew=2.0, zorder=6)
        dx, dy, ha, va = lab_off[s_]
        axB.annotate(f"{s_}\nas published", (x[j], y[j]),
                     textcoords="offset points", xytext=(dx, dy), ha=ha, va=va,
                     color=style["color"], fontsize=9.2,
                     fontweight="semibold", linespacing=1.4)
        axB.annotate("tight", (x[0], y[0]), textcoords="offset points",
                     xytext=(-7, 0), ha="right", va="center", color=MUTED,
                     fontsize=8.0)
        axB.annotate("loose", (x[-1], y[-1]), textcoords="offset points",
                     xytext=(7, 0), ha="left", va="center", color=MUTED,
                     fontsize=8.0)
    axB.set_xticks([10, 15, 20, 30, 50, 70])
    axB.set_xticklabels(["10", "15", "20", "30", "50", "70"])
    axB.minorticks_off()
    axB.set_xlabel("measured sky modes per 100 deg$^2$   "
                   "[$\\mathrm{tr}(WA)$ / patch area]")
    axB.set_ylabel(f"post-clean residual / HI   ({nmodes} modes removed)")
    axB.set_title("The tracks cross the line, they do not run along it",
                  color=INK2, fontsize=9.6, loc="left", pad=6)

    P._title(fig, "Loosening the prior buys modes that are worth less than nothing",
             f"Prior variance swept over {scales.min():g}$-${scales.max():g}$\\times$ "
             "the truth patch variance, everything else fixed (the prior MEAN was "
             "already flat; this is its amplitude). Scale 1 reproduces the published "
             "335.0/347.3 and 22.7/24.0 exactly. PROTOTYPE.",
             x=0.012, y_title=1.05, y_sub=0.982, override=text)
    fig.tight_layout(rect=(0, 0, 1, 0.91))
    return P._save(fig, "hi_priorvar")


fig_priorvar(pv)

## `hi_ladder`

*Sec. 4 -- How much sky does a strategy measure.* The drift's only geometry lever, out to N = 20, with the two measured floors overlaid on the middle panel.

Source: `ska_hi_design.fig_ladder`. Every `fig_*` takes `text=` to override the title/subtitle without touching the body, e.g.
`fig_ladder(lad, lad_long, floors, text=('My title', 'My subtitle'))`.

In [ ]:
def fig_ladder(ld, ld_long=None, floors=None, text=None):
    """What the drift's only geometry lever is worth, out to 20 dishes.

    Three panels:

      left    density against strip SPACING. Below one beam the strips overlap
              and densify the cross-scan direction a single drift strip cannot
              sample; at or above one beam they are disjoint and you buy area.
      middle  density against strip COUNT, now to N = 20, with N = 1 (a single
              parked dish) as the baseline.
      right   the decomposition, normalised to N = 1. The marginal return per
              strip is CONSTANT -- 11.7-12.7 modes for 43-47 deg² every time --
              so the density curve flattens purely because area grows linearly
              alongside modes, not because the ladder saturates.

    ``floors`` optionally overlays the two measured HI floors (3 and 12 dishes)
    on the middle panel, which is the only place mode count and residual appear
    together for the same geometry family.
    """
    n = np.asarray(ld["n_strips"], int)
    sp = np.asarray(ld["spacing_beams"], float)
    dens = np.asarray(ld["dens"], float)
    n_eff = np.asarray(ld["n_eff"], float)
    area = np.asarray(ld["area"], float)
    ref_n, ref_sp = int(ld["ref_count"]), float(ld["ref_spacing_beams"])

    # Merge the long sweep, so the count arm runs 1..20 rather than 1..8.
    if ld_long is not None:
        m = np.abs(np.asarray(ld_long["spacing_beams"], float) - ref_sp) < 1e-9
        for k, v, a, d in zip(np.asarray(ld_long["n_strips"], int)[m],
                              np.asarray(ld_long["n_eff"], float)[m],
                              np.asarray(ld_long["area"], float)[m],
                              np.asarray(ld_long["dens"], float)[m]):
            if k not in set(n[np.abs(sp - ref_sp) < 1e-9]):
                n = np.r_[n, k]; sp = np.r_[sp, ref_sp]
                n_eff = np.r_[n_eff, v]; area = np.r_[area, a]; dens = np.r_[dens, d]

    A = np.argsort(sp[np.abs(sp - sp) < 1])  # placeholder, replaced below
    iA = np.where(n == ref_n)[0][np.argsort(sp[n == ref_n])]
    iB = np.where(np.abs(sp - ref_sp) < 1e-9)[0]
    iB = iB[np.argsort(n[iB])]

    fig, axes = plt.subplots(1, 3, figsize=P._figsize("hi_ladder", (14.6, 4.9)),
                             gridspec_kw=dict(wspace=0.30))

    ax = axes[0]; P._tidy(ax)
    ax.plot(sp[iA], dens[iA], color=DRIFT, marker="o", ms=8, mfc=SURFACE,
            mew=2.0, zorder=4)
    j0 = int(np.where((n[iA] == ref_n) & (np.abs(sp[iA] - ref_sp) < 1e-9))[0][0])
    ax.plot([sp[iA][j0]], [dens[iA][j0]], color=DRIFT, marker="o", ms=13,
            mec=SURFACE, mew=2.0, zorder=5)
    ax.annotate("published\n52/50/48", (sp[iA][j0], dens[iA][j0]),
                textcoords="offset points", xytext=(0, 16), ha="center",
                color=DRIFT, fontsize=9.0, fontweight="semibold",
                linespacing=1.4)
    ax.axvline(1.0, color=INK, lw=1.4, ls=(0, (4, 3)), zorder=2)
    ax.annotate("one beam \u2014 strips stop overlapping", (1.0, dens[iA].min()),
                textcoords="offset points", xytext=(-6, 4), rotation=90,
                ha="right", va="bottom", color=INK, fontsize=8.4)
    ax.set_xlabel("strip spacing  [beam FWHM]")
    ax.set_ylabel("measured sky modes per 100 deg$^2$")
    ax.set_title(f"{ref_n} strips, varying spacing", color=INK2, fontsize=9.6,
                 loc="left", pad=6)

    ax = axes[1]; P._tidy(ax)
    ax.plot(n[iB], dens[iB], color=DRIFT, marker="o", ms=7, mfc=SURFACE,
            mew=2.0, zorder=4)
    j1 = int(np.where(n[iB] == 1)[0][0])
    ax.plot([1], [dens[iB][j1]], color=MUTED, marker="o", ms=11, mec=SURFACE,
            mew=1.8, zorder=5)
    ax.annotate("one parked dish\n(no ladder)", (1, dens[iB][j1]),
                textcoords="offset points", xytext=(10, -2), ha="left",
                va="top", color=MUTED, fontsize=8.6, linespacing=1.4)
    if floors:
        for nn, fl in floors.items():
            k = int(np.where(n[iB] == nn)[0][0])
            ax.plot([nn], [dens[iB][k]], color=DRIFT, marker="o", ms=13,
                    mec=SURFACE, mew=2.0, zorder=6)
            ax.annotate(f"{nn} dishes\nfloor {fl:.0f}$\\times$",
                        (nn, dens[iB][k]), textcoords="offset points",
                        xytext=(0, 15), ha="center", color=DRIFT,
                        fontsize=8.8, fontweight="semibold", linespacing=1.4)
    ax.set_xlabel("number of strips  (= dishes, observing simultaneously)")
    ax.set_ylabel("measured sky modes per 100 deg$^2$")
    ax.set_title(f"spacing fixed at {ref_sp:g} beams", color=INK2,
                 fontsize=9.6, loc="left", pad=6)

    ax = axes[2]; P._tidy(ax)
    ax.plot(n[iB], n_eff[iB] / n_eff[iB][j1], color=DRIFT, marker="o", ms=7,
            mfc=SURFACE, mew=2.0, zorder=4, label="measured modes, tr($WA$)")
    ax.plot(n[iB], area[iB] / area[iB][j1], color=THIRD, marker="s", ms=6,
            mfc=SURFACE, mew=2.0, zorder=4, label="sky area")
    ax.axhline(1.0, color=BASELINE, lw=1.0, ls=(0, (4, 3)), zorder=2)
    ax.set_xlabel("number of strips")
    ax.set_ylabel("relative to a single parked dish")
    ax.legend(loc="upper left", fontsize=8.8)
    ax.set_title("Constant marginal return: ~12 modes per 44 deg$^2$, every strip",
                 color=INK2, fontsize=9.6, loc="left", pad=6)

    P._title(fig, "The drift cannot cross-link, so the Dec ladder is its only "
             "geometry lever",
             "Mode count only, one channel at 350.8 MHz, no TOD. N strips on N "
             "sidereal days is arithmetically the same as N dishes parked at N "
             "elevations for one hour. The ladder does not saturate \u2014 but a "
             "mode count is not a residual, and cross-linking beats it 9$\\times$ "
             "at matched density.",
             x=0.012, y_title=1.05, y_sub=0.975, override=text)
    fig.tight_layout(rect=(0, 0, 1, 0.89))
    return P._save(fig, "hi_ladder")


fig_ladder(lad, lad_long, floors)

## `hi_transfer_function`

*Sec. 6 -- HI signal recovery.* T(k_par) per mode count, +/-1 sigma over 20 mocks.

Source: `ska_hi_plots.fig_transfer_function`. Every `fig_*` takes `text=` to override the title/subtitle without touching the body, e.g.
`fig_transfer_function(exp, text=('My title', 'My subtitle'))`.

In [ ]:
def fig_transfer_function(exp, var="cr_", text=None):
    """T(k_par). Defaults to the common-resolution, interior-pixel variant --
    the pipeline a real analysis would run."""
    nmodes = [int(n) for n in exp["nmodes_grid"]]
    k = exp[f"drift_{var}k"]
    fig, axes = plt.subplots(2, 4, figsize=_figsize("hi_transfer_function", (13.4, 6.4)), sharex=True, sharey=True)

    for ax, nm in zip(axes.ravel(), nmodes):
        _tidy(ax)
        ax.axhline(1.0, color=BASELINE, lw=1.0, ls=(0, (4, 3)), zorder=1)
        for s, style in STRAT.items():
            t = exp[f"{s}_{var}tf_{nm}"]
            e = exp[f"{s}_{var}tf_scatter_{nm}"]
            ax.fill_between(k, t - e, t + e, color=style["color"], alpha=0.16,
                            lw=0, zorder=2)
            ax.plot(k, t, color=style["color"], zorder=3,
                    solid_capstyle="round")
        ax.set_xscale("log")
        ax.set_title(f"{nm} mode{'s' if nm > 1 else ''} removed",
                     color=INK, pad=6)
        ax.set_ylim(-0.05, 1.15)
        ax.set_xlim(k.min() * 0.9, k.max() * 1.1)
        # The cell below the last top-row panel holds the legend, so that panel
        # has to carry its own x axis or the column loses it entirely.
        if ax is axes[0, -1]:
            ax.tick_params(labelbottom=True)
            ax.set_xlabel("$k_\\parallel$  [Mpc$^{-1}$]")

    # Direct labels on the first panel, so identity is not colour-alone.
    a0 = axes[0, 0]
    a0.annotate("drift", (k[7], exp[f"drift_{var}tf_{nmodes[0]}"][7]),
                textcoords="offset points", xytext=(0, -16),
                color=DRIFT, fontsize=9, fontweight="semibold", ha="center")
    a0.annotate("raster", (k[7], exp[f"raster_{var}tf_{nmodes[0]}"][7]),
                textcoords="offset points", xytext=(0, 8),
                color=RASTER, fontsize=9, fontweight="semibold", ha="center")

    # Spare cell carries the legend and the reading instruction.
    spare = axes.ravel()[len(nmodes)]
    spare.set_axis_off()
    spare.legend(handles=[Line2D([], [], color=v["color"], lw=2.4,
                                 label=v["label"]) for v in STRAT.values()]
                 + [Line2D([], [], color=BASELINE, lw=1.0, ls=(0, (4, 3)),
                           label="$T=1$, no signal lost")],
                 loc="upper left", bbox_to_anchor=(-0.02, 0.95),
                 handlelength=1.8, labelspacing=0.9)
    spare.text(-0.02, 0.30,
               "Band = $\\pm1\\sigma$ over 20\nindependent HI mocks.\n\n"
               "Loss concentrates at low\n$k_\\parallel$: those modes are the\n"
               "smoothest along the line of\nsight, so PCA mistakes them\n"
               "for foreground.",
               transform=spare.transAxes, va="top", color=INK2, fontsize=8.6,
               linespacing=1.5)

    for ax in axes[1, :len(nmodes) - 4]:
        ax.set_xlabel("$k_\\parallel$  [Mpc$^{-1}$]")
    # One shared y label — per-axes labels on a shared axis collide between rows.
    fig.supylabel("$T(k_\\parallel)$   —   fraction of HI power surviving the clean",
                  color=INK2, fontsize=10, x=0.005)

    _title(fig, "Cross-linking preserves HI through foreground cleaning at every scale",
             "350$-$400 MHz, 32 channels, nside 64. Channels reconvolved to a "
             "common 3.99$\\degree$ beam, 119 interior pixels. $T$ is each "
             "strategy's ratio against its own injected response, so it is "
             "depth-independent.",
           x=0.055, y_title=0.985, y_sub=0.938, override=text)
    fig.tight_layout(rect=(0, 0, 1, 0.925))
    return _save(fig, "hi_transfer_function")


fig_transfer_function(exp)

## `hi_ablation`

*Sec. 6 -- HI signal recovery.* Floor vs total vs noise. Common resolution since 2026-08-26 (tag='cr_').

Source: `ska_hi_plots.fig_ablation`. Every `fig_*` takes `text=` to override the title/subtitle without touching the body, e.g.
`fig_ablation(abl, text=('My title', 'My subtitle'))`.

In [ ]:
def fig_ablation(abl, text=None, tag="cr_"):
    """`tag` selects the convention: "cr_" (common resolution, interior pixels,
    matching figure 2), "int_" (as run, interior pixels) or "" (as run, full
    patch -- the originally published numbers). Older result files hold only
    the "" keys, so fall back to those rather than raising."""
    if f"drift_{tag}p_hi" not in abl:
        print(f"!! ablation results predate the '{tag}' variant -- "
              f"using as-run full-patch keys", flush=True)
        tag = ""
    nmodes = [int(n) for n in abl["nmodes_grid"]]
    # key, colour, label, linestyle, marker, markersize -- one marker per line
    # so the arms stay separable in greyscale and for CVD readers.
    #
    # FOURTH is categorical slot 4 of the validated palette (the next free slot:
    # slot 2, orange, is reserved for "raster" across this figure set, so using
    # it for a noise term inside a per-strategy panel would clash). Checked with
    # the skill's validator ported to Python -- blue/aqua/yellow on all pairs:
    # CVD dE 9.1 (target 8), normal-vision dE 22.9 (floor 15). Yellow's contrast
    # against the surface is 2.11, a WARN, which is why every line is legended
    # AND carries its own marker rather than resting on hue.
    FOURTH = "#eda100"
    arms = [("floor", DRIFT, "Beam + prior floor (noiseless data)", "-", "o", 5),
            ("total", INK, "Full data (floor + 1/f + white)", "none", "x", 9),
            ("gainonly", THIRD, "1/f gain alone", "-", "^", 6),
            ("whiteonly", FOURTH, "White noise alone", "-", "s", 5)]
    fig, axes = plt.subplots(1, 2, figsize=_figsize("hi_ablation", (11.6, 4.9)), sharey=True)

    for ax, s in zip(axes, ("drift", "raster")):
        _tidy(ax)
        p_hi = abl[f"{s}_{tag}p_hi"]
        for arm, colour, label, ls, mk, ms in arms:
            med = [np.median(np.sqrt(abl[f"{s}_{tag}{arm}_{nm}"] / p_hi))
                   for nm in nmodes]
            if arm == "total":
                # Markers only, over `floor`: the point is that they coincide.
                ax.plot(nmodes, med, ls="none", marker=mk, ms=ms,
                        mfc="none", mec=colour, mew=1.6, zorder=4)
            else:
                ax.plot(nmodes, med, color=colour, ls=ls, marker=mk, ms=ms,
                        mfc=SURFACE, mew=1.8, zorder=3)
        ax.axhline(1.0, color=INK, lw=1.4, zorder=2)
        ax.set_yscale("log")
        ax.set_xticks(nmodes); ax.set_xticklabels([str(n) for n in nmodes])
        ax.set_xlabel("PCA modes removed")
        ax.set_title(STRAT[s]["label"], color=STRAT[s]["color"], pad=6,
                     fontweight="semibold")
    axes[0].set_ylabel("cleaned residual / HI   (amplitude)")
    axes[0].annotate("HI level", (nmodes[0], 1.0), textcoords="offset points",
                     xytext=(2, 7), color=INK, fontsize=8.8)

    # Figure-level and horizontal: inside either panel it collides with the
    # noise curve or the HI-level line.
    fig.legend(handles=[Line2D([], [], color=c, ls=(ls if ls != "none" else "none"),
                               marker=mk, ms=ms,
                               mfc=("none" if a == "total" else SURFACE),
                               mec=c, mew=1.6, lw=(0 if ls == "none" else 2),
                               label=lab)
                        for a, c, lab, ls, mk, ms in arms],
               loc="upper left", bbox_to_anchor=(0.045, 0.90), ncols=4,
               handlelength=2.0, columnspacing=1.8)

    _title(fig, "The blocker is the floor, and 1/f is the smallest term in it",
             "Noiseless and full data track each other to a few per cent. Split "
             "apart, 1/f sits below the white noise, which is itself far below "
             "the floor — so neither is what limits HI recovery.",
           x=0.045, y_title=1.03, y_sub=0.965, override=text)
    fig.tight_layout(rect=(0, 0, 1, 0.84))
    return _save(fig, "hi_ablation")


fig_ablation(abl)

## `hi_cleaning_depth`

*Sec. 6 -- HI signal recovery.* Three bands. The net never turns over: there is no optimal cleaning depth.

Source: `ska_hi_design.fig_cleaning_depth`. Every `fig_*` takes `text=` to override the title/subtitle without touching the body, e.g.
`fig_cleaning_depth(, text=('My title', 'My subtitle'))`.

In [ ]:
def fig_cleaning_depth(text=None):
    """The first question asked of any floor-limited result: clean deeper?

    Removing more PCA modes lowers the residual and destroys more HI, and the
    two are not separable -- the floor is spectrally non-smooth precisely
    because the null space is chromatic, so it overlaps the signal. The useful
    quantity is therefore not the residual but the residual measured against
    the HI that SURVIVES the clean, ``ratio / T``. If that has a minimum, there
    is an optimal cleaning depth; if it falls monotonically, the limit is not
    the cleaning.

    Both bands, so the answer can be seen to be band-dependent or not.
    """
    runs = [("350$-$400", "hi_experiment_matched_f350_400_nc32_ns64.npz",
             ("drift", "drift12", "raster"), "-"),
            ("500$-$550", "hi_experiment_f500_f500_550_nc32_ns128.npz",
             ("drift", "raster"), (0, (1, 2.4))),
            ("675$-$725", "hi_experiment_f700_f675_725_nc32_ns128.npz",
             ("drift", "raster"), (0, (4, 2)))]
    colour = {"drift": DRIFT, "drift12": THIRD, "raster": RASTER}

    fig, (axA, axB) = plt.subplots(
        1, 2, figsize=P._figsize("hi_cleaning_depth", (12.6, 5.0)),
        gridspec_kw=dict(wspace=0.28))
    seen, strat_seen = [], []
    for band, fname, strategies, ls in runs:
        path = os.path.join(RESDIR, fname)
        if not os.path.exists(path):
            continue
        e = np.load(path)
        nm = [int(v) for v in e["nmodes_grid"]]
        for s_ in strategies:
            if f"{s_}_cr_tf_{nm[0]}" not in e:
                continue
            ratio = [float(np.median(np.sqrt(
                e[f"{s_}_cr_p_clean_auto_{k}"] / e[f"{s_}_cr_p_mapmade"])))
                for k in nm]
            tf = [float(np.median(e[f"{s_}_cr_tf_{k}"])) for k in nm]
            net = [r / t for r, t in zip(ratio, tf)]
            c = colour[s_]
            axA.plot(nm, ratio, color=c, ls=ls, marker="o", ms=6, mfc=SURFACE,
                     mew=1.8, zorder=4)
            axB.plot(nm, net, color=c, ls=ls, marker="o", ms=6, mfc=SURFACE,
                     mew=1.8, zorder=4)
            seen.append((band, ls))
            if s_ not in [t[0] for t in strat_seen]:
                strat_seen.append((s_, c))
    for ax, ylab, title in (
            (axA, f"post-clean residual / HI",
             "What deeper cleaning buys"),
            (axB, "residual / HI that SURVIVES the clean",
             "What it costs — and whether it is worth it")):
        P._tidy(ax); ax.set_yscale("log")
        ax.axhline(1.0, color=INK, lw=1.5, zorder=3)
        ax.annotate("HI level", (12.0, 1.0), textcoords="offset points",
                    xytext=(0, 6), ha="right", color=INK, fontsize=8.6)
        ax.set_xlabel("PCA modes removed")
        ax.set_ylabel(ylab, fontsize=9.4)
        ax.set_title(title, color=INK2, fontsize=9.6, loc="left", pad=6)
        ax.set_xticks([1, 2, 3, 4, 6, 8, 10])
        ax.set_xlim(0.5, 12.2)
        ax.set_ylim(bottom=0.7)
    bands = list(dict.fromkeys(seen))
    axA.legend(handles=[Line2D([], [], color=c, lw=2.4, label=s_)
                        for s_, c in strat_seen],
               loc="upper right", bbox_to_anchor=(1.0, 1.0),
               fontsize=8.8, title="strategy", title_fontsize=8.4,
               labelspacing=0.35)
    axB.legend(handles=[Line2D([], [], color=MUTED, ls=ls, lw=2.0,
                               label=f"{b} MHz") for b, ls in bands],
               loc="lower left", bbox_to_anchor=(0.02, 0.04),
               fontsize=8.8, title="band", title_fontsize=8.4,
               labelspacing=0.35)

    P._title(fig, "Cleaning harder always helps — which is the point",
             "The residual keeps falling with mode count and so does the HI, "
             "but the residual falls faster, so the net never turns over within "
             "the range tested. There is no optimal depth to find: the limit is "
             "the beam + prior floor, not the cleaning. Common resolution, "
             "interior pixels.",
             x=0.012, y_title=1.05, y_sub=0.975, override=text)
    fig.tight_layout(rect=(0, 0, 1, 0.90))
    return P._save(fig, "hi_cleaning_depth")


fig_cleaning_depth()

## `hi_realisations`

*Sec. 6 -- HI signal recovery.* Nine HI realisations. The paired ratio is tighter than either absolute.

Source: `ska_hi_design.fig_realisations`. Every `fig_*` takes `text=` to override the title/subtitle without touching the body, e.g.
`fig_realisations(vals, text=('My title', 'My subtitle'))`.

In [ ]:
def fig_realisations(vals, text=None):
    """How much of a single-realisation number is the realisation?

    Left: every run, per strategy, with mean and +/-1 sd. Right: the PAIRED
    drift/raster ratio, which is tighter than either absolute because the
    realisation partly cancels when both strategies see the same HI field.
    That is the practical conclusion -- quote ratios between strategies, not
    absolute residuals.
    """
    dr, ra = vals["drift"], vals["raster"]
    if len(dr) < 3:
        return None
    fig, (axA, axB) = plt.subplots(
        1, 2, figsize=P._figsize("hi_realisations", (11.8, 4.8)),
        gridspec_kw=dict(width_ratios=[1.35, 1.0], wspace=0.30))

    P._tidy(axA); axA.set_yscale("log")
    rng = np.random.default_rng(0)
    for i, (s_, v) in enumerate((("drift", dr), ("raster", ra))):
        c = STRAT[s_]["color"]
        x = i + 1 + rng.uniform(-0.09, 0.09, len(v))
        axA.plot(x, v, ls="none", marker="o", ms=7, mfc=SURFACE, mec=c,
                 mew=1.8, zorder=4)
        m, sd = v.mean(), v.std(ddof=1)
        axA.plot([i + 1 - 0.28, i + 1 + 0.28], [m, m], color=c, lw=2.6,
                 zorder=5)
        axA.add_patch(Rectangle((i + 1 - 0.28, m - sd), 0.56, 2 * sd,
                                facecolor=c, alpha=0.13, edgecolor="none",
                                zorder=2))
        axA.annotate(f"{m:.0f} $\\pm$ {sd:.0f}\n({100 * sd / m:.1f}%)",
                     (i + 1 + 0.33, m), va="center", ha="left", color=c,
                     fontsize=9.2, fontweight="semibold", linespacing=1.4)
        axA.plot([i + 1], [v[0]], marker="*", ms=14, color=INK, zorder=6)
    axA.annotate("$\\star$ = the published run", (0.03, 0.05),
                 xycoords="axes fraction", color=INK, fontsize=8.6)
    axA.set_xlim(0.55, 2.75); axA.set_xticks([1, 2])
    axA.set_xticklabels(["drift", "raster"])
    axA.set_ylabel(f"post-clean residual / HI   ({NMODES} modes removed)")
    axA.set_title(f"{len(dr)} HI realisations, same pixels, same noise seeds",
                  color=INK2, fontsize=9.6, loc="left", pad=6)

    P._tidy(axB)
    r = dr / ra
    x = 1 + rng.uniform(-0.09, 0.09, len(r))
    axB.plot(x, r, ls="none", marker="o", ms=7, mfc=SURFACE, mec=THIRD,
             mew=1.8, zorder=4)
    m, sd = r.mean(), r.std(ddof=1)
    axB.plot([0.72, 1.28], [m, m], color=THIRD, lw=2.6, zorder=5)
    axB.add_patch(Rectangle((0.72, m - sd), 0.56, 2 * sd, facecolor=THIRD,
                            alpha=0.13, edgecolor="none", zorder=2))
    axB.annotate(f"{m:.1f} $\\pm$ {sd:.1f}\n({100 * sd / m:.1f}%)",
                 (1.35, m), va="center", ha="left", color=THIRD, fontsize=9.6,
                 fontweight="semibold", linespacing=1.4)
    axB.plot([1], [r[0]], marker="*", ms=14, color=INK, zorder=6)
    axB.set_xlim(0.5, 2.1); axB.set_xticks([1])
    axB.set_xticklabels(["drift / raster"])
    axB.set_ylabel("ratio between strategies")
    axB.set_title("The paired ratio is tighter than either number",
                  color=INK2, fontsize=9.6, loc="left", pad=6)

    P._title(fig, "A single-realisation residual carries a 9$-$13% error bar",
             "Only the true HI seed varies; noise seeds and pixels are fixed, so "
             "this is realisation scatter alone. It is the same size as several "
             "effects this series has reported, which is why the cost-matched "
             "ladder test came out inconclusive. Quote ratios, not absolutes.",
             x=0.012, y_title=1.05, y_sub=0.975, override=text)
    fig.tight_layout(rect=(0, 0, 1, 0.89))
    return P._save(fig, "hi_realisations")


fig_realisations(vals)

## `hi_chromatic`

*Sec. 7 -- Why a different cleaning basis cannot help.* Rank, eigenvectors and the principal angles between floor and HI. Replaces the old hi_rank + hi_eigenvectors pair.

Source: `ska_hi_design.fig_chromatic`. Every `fig_*` takes `text=` to override the title/subtitle without touching the body, e.g.
`fig_chromatic(rk, text=('My title', 'My subtitle'))`.

In [ ]:
def fig_chromatic(rank, text=None):
    """The physical heart of the result, in three panels.

    Foreground cleaning works by assuming the contaminant is spectrally smooth
    and therefore low-rank in frequency. That is true of the foreground itself,
    and it is why one PCA mode removes it. It is NOT true of the beam + prior
    floor: the floor is ``(I - WA)s``, and ``WA`` is chromatic because the beam
    (and hence the measured subspace) changes across the band. The floor
    therefore inherits spectral structure the sky never had.

      left    rank. The foreground is rank 1. The floor needs 2-6 modes. The HI
              needs 23-25 of 32 -- it is the one component that is genuinely
              not compressible.
      middle  the eigenvectors themselves, which is what "chromatic" means
              concretely: the foreground's first mode is a clean power law, the
              floor's get progressively more structured, and the HI's are
              noise-like from the start.
      right   principal angles between the floor's subspace and the HI's. They
              are not orthogonal, so projecting out the floor necessarily takes
              HI with it. This is why deeper cleaning cannot win.
    """
    f = np.asarray(rank["freqs"], float)
    arms = [("fg", "foreground", INK), ("floor", "beam + prior floor", DRIFT),
            ("hi", "HI", RASTER)]
    fig, axes = plt.subplots(1, 3, figsize=P._figsize("hi_chromatic", (14.4, 4.7)),
                             gridspec_kw=dict(wspace=0.30))

    # --- left: how many modes each component occupies ----------------------
    ax = axes[0]; P._tidy(ax)
    for s_, ls in (("drift", "-"), ("raster", (0, (4, 2)))):
        for arm, lab, c in arms:
            w = np.asarray(rank[f"{s_}_w_{arm}"], float)
            cum = np.cumsum(w) / w.sum()
            ax.plot(np.arange(1, len(cum) + 1), cum, color=c, ls=ls, lw=2.0,
                    zorder=4, label=lab if s_ == "drift" else None)
    ax.axhline(0.99, color=BASELINE, lw=1.0, ls=(0, (4, 3)), zorder=2)
    ax.annotate("99% of variance", (len(f), 0.99), textcoords="offset points",
                xytext=(-2, -12), ha="right", color=MUTED, fontsize=8.4)
    ax.set_xlim(1, 30); ax.set_ylim(0.3, 1.02)
    ax.set_xlabel("eigenmodes retained")
    ax.set_ylabel("cumulative fraction of variance")
    leg = ax.legend(loc="lower right", fontsize=8.8)
    ax.add_artist(leg)
    ax.legend(handles=[Line2D([], [], color=MUTED, ls="-", lw=2.0,
                              label="drift"),
                       Line2D([], [], color=MUTED, ls=(0, (4, 2)), lw=2.0,
                              label="raster")],
              loc="center right", fontsize=8.4, labelspacing=0.3)
    ax.set_title("The HI is the only thing that is not low-rank",
                 color=INK2, fontsize=9.6, loc="left", pad=6)

    # --- middle: the spectral shapes ---------------------------------------
    ax = axes[1]; P._tidy(ax)
    V = {a: np.asarray(rank[f"drift_V_{a}"], float) for a, _, _ in arms}
    def norm(v):
        v = v - v.mean()
        return v / (np.abs(v).max() or 1.0)
    ax.plot(f, norm(V["fg"][:, 0]), color=INK, lw=2.4, zorder=5,
            label="foreground, mode 1")
    for k, alpha in zip(range(3), (1.0, 0.65, 0.42)):
        ax.plot(f, norm(V["floor"][:, k]) + 2.4 * (k + 1), color=DRIFT, lw=2.0,
                alpha=alpha, zorder=4,
                label="floor, modes 1$-$3" if k == 0 else None)
    ax.plot(f, norm(V["hi"][:, 0]) + 2.4 * 4, color=RASTER, lw=1.6, zorder=4,
            label="HI, mode 1")
    ax.set_yticks([]); ax.set_xlabel("frequency  [MHz]")
    ax.set_ylabel("eigenvector  (offset, normalised)")
    ax.set_xlim(f[0] - 1, f[-1] + 17)
    for y0, lab, c in ((0.0, "foreground\nmode 1", INK),
                       (2.4, "floor\nmodes 1$-$3", DRIFT),
                       (2.4 * 4, "HI\nmode 1", RASTER)):
        ax.annotate(lab, (f[-1] + 1.5, y0), ha="left", va="center", color=c,
                    fontsize=8.4, fontweight="semibold", linespacing=1.35)
    ax.set_title("Smooth, then structured, then noise-like",
                 color=INK2, fontsize=9.6, loc="left", pad=6)

    # --- right: the floor and the HI are not orthogonal ---------------------
    ax = axes[2]; P._tidy(ax)
    for s_, style in STRAT.items():
        c = np.asarray(rank[f"{s_}_overlap"], float)
        ax.plot(np.arange(1, len(c) + 1), c, color=style["color"], marker="o",
                ms=6, mfc=SURFACE, mew=1.8, zorder=4, label=s_)
    ax.axhline(0.0, color=INK, lw=1.4, zorder=3)
    ax.annotate("0 would mean the floor could be removed\nwithout touching the HI",
                (1, 0.06), color=MUTED, fontsize=8.4, ha="left", va="bottom",
                linespacing=1.45)
    ax.set_ylim(-0.05, 1.0)
    ax.set_xlabel("principal angle index")
    ax.set_ylabel("cos(principal angle), floor vs HI")
    ax.legend(loc="upper left", fontsize=8.8)
    ax.set_title("They overlap, so cleaning one costs the other",
                 color=INK2, fontsize=9.6, loc="left", pad=6)

    P._title(fig, "The floor is low-rank but not spectrally smooth — which "
             "is why PCA cannot remove it",
             "Foreground cleaning assumes the contaminant is smooth in "
             "frequency. The foreground is (rank 1). The beam + prior floor is "
             "not: $WA$ is chromatic, so the floor carries structure the sky "
             "never had, and its subspace overlaps the HI's at "
             "cos $=0.42-0.78$. 350$-$400 MHz.",
             x=0.012, y_title=1.05, y_sub=0.975, override=text)
    fig.tight_layout(rect=(0, 0, 1, 0.89))
    return P._save(fig, "hi_chromatic")


fig_chromatic(rk)

## `hi_design`

*Sec. 8 -- Survey design (headline).* Two regimes, and the lever chart including the dead levers.

Source: `ska_hi_design.fig_design`. Every `fig_*` takes `text=` to override the title/subtitle without touching the body, e.g.
`fig_design(d, text=('My title', 'My subtitle'))`.

In [ ]:
def _levers(d):
    dr, d12, ra = d["drift"], d["drift12"], d["raster"]
    return [
        ("Cross-linking (at matched density)",
         ra["floor"] / d12["floor"], "measured",
         f"drift12 vs raster, {d12['dens']:.1f} vs {ra['dens']:.1f} "
         "modes/100 deg$^2$ \u2014 a step change, not a slope"),
        ("Declination ladder, 3 $\\rightarrow$ 12 dishes",
         d12["floor"] / dr["floor"], "measured",
         f"12 dish-hours against 3; floor slope ${d['slope_ladder']:.2f}$, "
         "linear over 3$-$12"),
        ("Frequency, 350 $\\rightarrow$ 700 MHz (drift)", 1.0 / 105.0,
         "measured",
         "foreground residual only; the HI is 4.3$\\times$ fainter there, so "
         "the RATIO gains 24$\\times$"),
        ("Frequency, 350 $\\rightarrow$ 700 MHz (raster)", 1.0 / 2.5,
         "measured",
         "the raster barely gains \u2014 it was already well conditioned"),
        ("Ladder spacing, 0.5 $\\rightarrow$ 0.25 beams", None, "ambiguous",
         "cost-matched; sign flips with the metric, so not a result"),
        ("Integration time", 1.0, "dead", "adds samples, not rows of $A$"),
        ("System temperature", 1.0, "dead",
         "the floor is 96$-$99% of the residual"),
        ("Co-pointed dishes", 1.0, "dead",
         "same statement as integration time"),
    ]


def _panel_levers(ax, d):
    P._tidy(ax)
    rows = _levers(d)
    y = np.arange(len(rows))[::-1]
    xlo, xhi = 0.005, 3.0
    ax.set_xscale("log")
    ax.set_xlim(xlo, xhi)
    ax.set_ylim(-0.8, len(rows) - 0.3)
    ax.axvline(1.0, color=INK, lw=1.6, zorder=4)
    ax.grid(axis="y", visible=False)

    for yi, (label, factor, kind, note) in zip(y, rows):
        if kind == "measured":
            ax.barh(yi, factor - 1.0, left=1.0, height=0.32, color=THIRD,
                    zorder=3)
            txt = f"$\\times${factor:.2f}" if factor >= 0.1 else \
                  f"$\\times${factor:.3f}"
            ax.annotate(txt, (factor, yi), textcoords="offset points",
                        xytext=(6, 0), ha="left", va="center", color=INK,
                        fontsize=9.4, fontweight="semibold",
                        bbox=dict(facecolor=SURFACE, edgecolor="none", pad=1.0))
        elif kind in ("open", "ambiguous"):
            ax.add_patch(Rectangle((xlo, yi - 0.16), 1.0 - xlo, 0.32,
                                   facecolor="none", edgecolor=BASELINE,
                                   lw=1.1, hatch="////", zorder=3))
            ax.annotate("inconclusive" if kind == "ambiguous" else "not measured",
                        (0.8, yi), va="center", ha="right", color=MUTED,
                        fontsize=8.4, style="italic", zorder=5,
                        bbox=dict(facecolor=SURFACE, edgecolor="none", pad=1.5))
        else:
            ax.plot([1.0], [yi], marker="o", ms=9, color=MUTED, mec=SURFACE,
                    mew=1.6, zorder=5)
            ax.annotate("1.00$\\times$ exactly", (1.12, yi), va="center",
                        color=MUTED, fontsize=8.6)
        ax.annotate(note, (xlo * 1.15, yi - 0.30), color=MUTED, fontsize=7.3,
                    va="center", ha="left")

    ax.set_yticks(y)
    ax.set_yticklabels([r[0] for r in rows], fontsize=8.8)
    for tick, (_l, _f, kind, _n) in zip(ax.get_yticklabels(), rows):
        tick.set_color(MUTED if kind == "dead" else INK)
    ax.set_xticks([0.01, 0.05, 0.2, 1.0])
    ax.set_xticklabels(["0.01", "0.05", "0.2", "1$\\times$"])
    ax.minorticks_off()
    ax.set_xlabel("factor change in the beam + prior floor   (left is better)")
    ax.set_title("Frequency is the biggest lever, and it is band-dependent",
                 color=INK2, fontsize=9.6, loc="left", pad=6)


def _panel_tradespace(ax, d):
    P._tidy(ax)
    ax.set_xscale("log"); ax.set_yscale("log")
    xlo, xhi, ylo, yhi = 11.0, 30.0, 0.55, 1200.0
    ax.set_xlim(xlo, xhi); ax.set_ylim(ylo, yhi)

    dr, d12, ra = d["drift"], d["drift12"], d["raster"]

    # --- the ladder family: measured, one run, slope from two of its points --
    xs = np.geomspace(xlo, xhi, 200)
    ys = dr["floor"] * (xs / dr["dens"]) ** d["slope_ladder"]
    seg = (xs >= dr["dens"]) & (xs <= d12["dens"])
    ax.plot(xs, ys, color=DRIFT, lw=1.2, ls=(0, (5, 3)), alpha=0.6, zorder=2)
    ax.plot(xs[seg], ys[seg], color=DRIFT, lw=2.8, zorder=3)

    # drift8 comes from the OTHER run, so it is drawn open and said so.
    if "drift8" in d:
        d8 = d["drift8"]
        ax.plot([d8["dens"]], [d8["floor"]], marker="o", ms=10, mfc=SURFACE,
                mec=DRIFT, mew=2.2, zorder=5)
        ax.annotate("8 dishes\n(other run)", (d8["dens"], d8["floor"]),
                    textcoords="offset points", xytext=(-9, -6), ha="right",
                    va="top", color=MUTED, fontsize=8.2, linespacing=1.4)

    for k, lab, off in (("drift", "3 dishes", (-10, 4)),
                        ("drift12", "12 dishes", (12, 12))):
        ax.plot([d[k]["dens"]], [d[k]["floor"]], marker="o", ms=13,
                color=DRIFT, mec=SURFACE, mew=2.0, zorder=6)
        ax.annotate(lab, (d[k]["dens"], d[k]["floor"]),
                    textcoords="offset points", xytext=off,
                    ha="right" if off[0] < -8 else "left",
                    color=DRIFT, fontsize=9.4, fontweight="semibold")
    ax.annotate(f"parked drift, adding dishes\nslope ${d['slope_ladder']:.2f}$",
                (xlo * 1.04, 110.0), color=DRIFT, fontsize=8.8, ha="left",
                va="center", linespacing=1.45)

    # --- the raster, essentially the same mode density, a decade lower -------
    ax.plot([ra["dens"]], [ra["floor"]], marker="o", ms=13, color=RASTER,
            mec=SURFACE, mew=2.0, zorder=6)
    ax.annotate("raster\ncross-linked", (ra["dens"], ra["floor"]),
                textcoords="offset points", xytext=(10, -2), ha="left",
                va="top", color=RASTER, fontsize=9.4, fontweight="semibold",
                linespacing=1.4)
    # The measured gap. Both endpoints are data; nothing here is extrapolated.
    xbar = np.sqrt(d12["dens"] * ra["dens"])
    ax.plot([xbar, xbar], [ra["floor"], d12["floor"]], color=INK, lw=1.8,
            zorder=4)
    for y in (ra["floor"], d12["floor"]):
        ax.plot([xbar * 0.985, xbar * 1.015], [y, y], color=INK, lw=1.8,
                zorder=4)
    ax.annotate(f"$\\times${d['crosslink_gain']:.0f}\nat the same\nmode density",
                (xbar * 0.97, np.sqrt(ra["floor"] * d12["floor"])), ha="right",
                va="center", color=INK, fontsize=9.6, fontweight="semibold",
                linespacing=1.45)
    ax.annotate(f"{d12['dens']:.1f} vs {ra['dens']:.1f} modes/100 deg$^2$ "
                "\u2014 2% apart",
                (xbar, ylo * 2.2), ha="center", va="center", color=MUTED,
                fontsize=8.2)

    ax.axhline(1.0, color=INK, lw=1.6, zorder=3)
    ax.annotate("HI level", (xlo * 1.04, 1.0), textcoords="offset points",
                xytext=(0, 6), color=INK, fontsize=8.8)

    ax.set_xticks([12, 15, 20, 25, 30])
    ax.set_xticklabels(["12", "15", "20", "25", "30"])
    ax.minorticks_off()
    ax.set_xlabel("measured sky modes per 100 deg$^2$   "
                  "[$\\mathrm{tr}(WA)$ / patch area]")
    ax.set_ylabel(f"beam + prior floor / HI   ({NMODES} modes removed)")
    ax.set_title("Two regimes, measured \u2014 350$-$400 MHz", color=INK2,
                 fontsize=9.6, loc="left", pad=6)


def fig_design(d, text=None):
    fig, (axA, axB) = plt.subplots(
        1, 2, figsize=P._figsize("hi_design", (14.8, 6.1)),
        gridspec_kw=dict(width_ratios=[1.25, 1.0], wspace=0.52))
    _panel_tradespace(axA, d)
    _panel_levers(axB, d)
    P._title(fig,
             "Cross-linking is not more modes \u2014 it is a different regime",
             "Beam + prior floor at 4 modes removed, common resolution, 119 "
             "interior pixels. Filled points share one HI realisation, one pixel "
             "set and one set of noise seeds; the open point is a different run "
             "(~15% realisation scatter). Mode density is tr($WA$) per unit sky.",
             x=0.012, y_title=1.04, y_sub=0.975, override=text)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    return P._save(fig, "hi_design")


fig_design(d)

## `hi_frequency_design`

*Sec. 8 -- Survey design.* Instrument, cosmology and their misleading quotient, over three bands.

Source: `ska_hi_design.fig_frequency`. Every `fig_*` takes `text=` to override the title/subtitle without touching the body, e.g.
`fig_frequency(bands, text=('My title', 'My subtitle'))`.

In [ ]:
def fig_frequency(bands, text=None):
    """Three panels: the numerator, the denominator, and why their ratio lies.

    ``residual/HI`` is the quantity this series quotes throughout, and within a
    band it is the right one. Across bands it is not: the HI's own brightness
    falls with redshift, so the denominator moves for reasons that have nothing
    to do with the instrument. Plotting the two factors separately is the only
    honest way to show a frequency dependence.
    """
    if len(bands) < 2:
        return None
    x = [b["fc"] for b in bands]
    fig, axes = plt.subplots(1, 3, figsize=P._figsize("hi_frequency_design",
                                                      (14.2, 4.6)),
                             gridspec_kw=dict(wspace=0.32))

    panels = [
        ("fg", "post-clean foreground residual", "what the INSTRUMENT does"),
        (None, "HI amplitude  $\\sqrt{P_{\\mathrm{true}}}$",
         "what COSMOLOGY does"),
        ("ratio", f"residual / HI   ({NMODES} modes removed)",
         "their quotient — and it inverts"),
    ]
    for ax, (key, ylab, title) in zip(axes, panels):
        P._tidy(ax); ax.set_yscale("log")
        if key is None:
            y = [b["hi"] for b in bands]
            ax.plot(x, y, color=INK, marker="o", ms=8, mfc=SURFACE, mew=2.0,
                    zorder=4)
            ax.annotate(f"{y[0] / y[-1]:.1f}$\\times$ fainter\nacross this range",
                        (x[0], y[-1]), textcoords="offset points",
                        xytext=(8, 0), ha="left", va="center", color=INK,
                        fontsize=8.8, linespacing=1.45)
        else:
            for s_, style in STRAT.items():
                if s_ not in bands[0]:
                    continue
                y = [b[s_][key] for b in bands]
                ax.plot(x, y, color=style["color"], marker="o", ms=8,
                        mfc=SURFACE, mew=2.0, zorder=4)
                ax.annotate(s_, (x[-1], y[-1]), textcoords="offset points",
                            xytext=(8, 0), va="center", color=style["color"],
                            fontsize=9.4, fontweight="semibold")
                fac = y[0] / y[-1]
                ax.annotate(f"{fac:.0f}$\\times$" if fac >= 10
                            else f"{fac:.1f}$\\times$",
                            (x[-1], y[-1]), textcoords="offset points",
                            xytext=(8, 13), va="center", color=MUTED,
                            fontsize=8.4)
            if key == "ratio":
                ax.axhline(1.0, color=INK, lw=1.4, zorder=3)
                ax.annotate("HI level", (x[0], 1.0),
                            textcoords="offset points", xytext=(2, 6),
                            color=INK, fontsize=8.4)
        ax.set_xlabel("band centre  [MHz]")
        ax.set_ylabel(ylab, fontsize=9.2)
        ax.set_title(title, color=INK2, fontsize=9.6, loc="left", pad=6)
        ax.set_xticks(x); ax.set_xticklabels([b["label"] for b in bands],
                                             fontsize=8.4)
        ax.set_xlim(min(x) - 60, max(x) + 110)
        ax.margins(y=0.18)

    P._title(fig, "residual / HI is not comparable across bands",
             "Left: the drift's foreground residual improves far faster with "
             "frequency than the raster's. Middle: the HI itself dims with "
             "redshift, which is cosmology, not instrument. Right: dividing one "
             "by the other inverts the ordering — so quote the two factors "
             "separately whenever bands are compared.",
             x=0.012, y_title=1.06, y_sub=0.975, override=text)
    fig.tight_layout(rect=(0, 0, 1, 0.88))
    return P._save(fig, "hi_frequency_design")


fig_frequency(bands)